# G1 Locomotion -- Colab Training (Phase 2)

Run cells in order. Runtime > Change runtime type > **T4 GPU** before starting.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = '/content/drive/MyDrive/g1_checkpoints'  # survives disconnects; edit if you want a different path

Mounted at /content/drive


In [2]:
!git clone https://github.com/shubhamt2897/Humanoid_LocomotioN.git /content/g1_locomotion
%cd /content/g1_locomotion

Cloning into '/content/g1_locomotion'...
remote: Enumerating objects: 140, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 140 (delta 34), reused 124 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (140/140), 17.50 MiB | 18.01 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/g1_locomotion


In [3]:
# Deliberately NOT installing torch -- Colab ships a CUDA-matched build preinstalled.
# requirements.txt pins torch too (for local/CPU use); skip that line here.
!pip install -q mujoco==3.12.0 rsl-rl-lib==5.5.0 tensordict==0.14.0 wandb==0.29.0 onnx==1.22.0

import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'CUDA not available -- check Runtime > Change runtime type > T4 GPU'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.7/232.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.0/594.0 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 18.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not 

In [4]:
import wandb
wandb.login()  # paste your API key from wandb.ai/authorize when prompted, once per Colab session

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: shubhamt2897 (shubhamt2897-hochschule-schmalkalden) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
RUN_NAME = 'asymmetric_payload_run'

# First run (nothing to resume from yet):
!python train.py \
  --num_envs 64 --device cuda --iterations 500 \
  --save_interval 100 \
  --wandb --run_name {RUN_NAME} \
  --log_dir {CHECKPOINT_DIR}/{RUN_NAME}


Streaming output truncated to the last 5000 lines.
                           Time elapsed: 0:09:43
                                    ETA: 0:05:57

################################################################################
                           Learning iteration 310/500                            

                            Total steps: 477696 
                       Steps per second: 806 
                        Collection time: 1.777s 
                          Learning time: 0.127s 
                        Mean value loss: 9.7411
                    Mean surrogate loss: -0.0322
                      Mean entropy loss: 42.1407
                            Mean reward: 11.93
                    Mean episode length: 45.96
                        Mean action std: 1.04
                reward/lin_vel_tracking: 0.4630
                reward/ang_vel_tracking: 0.0615
                  reward/contact_timing: 0.0000
                   reward/double_flight: -0.0209
              

## Resuming after a disconnect

Colab free tier can drop the session mid-run without warning. Because `--log_dir` points at
Drive, whatever was already checkpointed (every `--save_interval` iterations) is safe. To
resume: reconnect, re-run the mount/clone/install/login cells above, then run the cell below
instead of the first training cell -- pick the highest `model_<N>.pt` actually present in your
Drive checkpoint folder.

**Note:** `--iterations` here means "how many *more* iterations to run from the checkpoint",
not an absolute target -- `runner.load()` restores the saved iteration count, and `learn()` adds
`--iterations` on top of that.

In [ ]:
RUN_NAME = 'asymmetric_payload_run'
RESUME_FROM = f'{CHECKPOINT_DIR}/{RUN_NAME}/model_499.pt'

!python train.py \
  --num_envs 64 --device cuda --iterations 4500 \
  --save_interval 250 \
  --wandb --run_name {RUN_NAME} \
  --log_dir {CHECKPOINT_DIR}/{RUN_NAME} \
  --resume {RESUME_FROM}


Streaming output truncated to the last 5000 lines.
             reward/action_rate_penalty: -0.0320
                     reward/alive_bonus: 0.1000
--------------------------------------------------------------------------------
                         Iteration time: 2.77s
                           Time elapsed: 0:51:48
                                    ETA: 1:49:51

################################################################################
                          Learning iteration 1941/4999                           

                            Total steps: 2216448 
                       Steps per second: 586 
                        Collection time: 2.457s 
                          Learning time: 0.163s 
                        Mean value loss: 9.3109
                    Mean surrogate loss: -0.0208
                      Mean entropy loss: 47.5185
                            Mean reward: 42.18
                    Mean episode length: 65.34
                        Mea

## Export the final policy to ONNX (also written straight to Drive)

In [ ]:
# NOTE: with --iterations 1500 (0-indexed loop), the LAST checkpoint is model_1499.pt, not
# model_1500.pt -- rsl_rl's runner saves every --save_interval iterations plus one final save
# at whatever iteration the loop actually stopped on. Check {CHECKPOINT_DIR}/{RUN_NAME}/ and use
# whichever model_<N>.pt is actually the highest N present.
!python export_policy.py \
  --checkpoint {CHECKPOINT_DIR}/{RUN_NAME}/model_4999.pt \
  --out {CHECKPOINT_DIR}/{RUN_NAME}/g1_policy.onnx